# singular-matrix-mask-trick — worked example 1: Detect and replace two singular matrices in a batch of five

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `singular-matrix-mask-trick`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When solving a batch of linear systems, some matrices may be singular (determinant ≈ 0), which would cause `torch.linalg.solve` to raise an error or return garbage. The standard fix is: detect singular slices with `t.linalg.det`, clone the batch, overwrite singular slices with the identity matrix, then solve. The final step is masking out the results for singular slices.

## Worked solution

**Step 1 — Build a batch with two known singular matrices.** A singular matrix has at least one row that is a multiple of another. We make two rank-deficient 2×2 matrices and mix them into a batch of 5.

**Step 2 — Compute determinants.** `t.linalg.det(A)` returns a (K,) tensor of determinants, one per slice.

**Step 3 — Build the singularity mask.** `is_singular = dets.abs() < eps`. For exact singular matrices, det = 0.0.

**Step 4 — Clone and patch.** We must clone A first to avoid mutating the original. Then `A_safe[is_singular] = t.eye(n)` overwrites the bad slices with identity matrices that are trivially solvable.

**Step 5 — Solve and return validity.** `~is_singular` tells the caller which solutions are meaningful.

In [ ]:
import torch as t

t.manual_seed(42)

def solve_with_mask(A, b, eps=1e-8):
    K, n, _ = A.shape
    dets = t.linalg.det(A)
    is_singular = dets.abs() < eps
    A_safe = A.clone()
    A_safe[is_singular] = t.eye(n, dtype=A.dtype)
    x = t.linalg.solve(A_safe, b)
    return x, ~is_singular

# Build a batch of 5 systems; indices 1 and 3 are singular
A = t.zeros(5, 2, 2)
A[0] = t.tensor([[2.0, 1.0], [0.0, 3.0]])
A[1] = t.tensor([[1.0, 2.0], [2.0, 4.0]])   # singular: row1 = 2*row0
A[2] = t.tensor([[3.0, 0.0], [1.0, 2.0]])
A[3] = t.tensor([[0.0, 0.0], [1.0, 1.0]])   # singular: first row is zero
A[4] = t.tensor([[1.0, 1.0], [0.0, 1.0]])
b = t.randn(5, 2)

x, is_valid = solve_with_mask(A, b)
print('Valid slices:', is_valid.tolist())  # [T, F, T, F, T]
print('det[1]:', t.linalg.det(A[1]).item())
print('det[3]:', t.linalg.det(A[3]).item())
print('Solutions for valid slices:', x[is_valid].shape)